In [4]:
"""
Tennis dataset cleaning pipeline — remaining 7 tables
(statistics, pbp, power, odds, venue, votes, season)

Focus: high-quality cleaning especially for statistics & odds.
Requires: pandas, numpy, pyarrow (or fastparquet)
"""

import pandas as pd
import numpy as np
import os

In [5]:
os.getcwd()

'D:\\Tennis Project\\Tennis Schema\\tennis_project'

In [6]:
# ---------------------------------------------------------------------------
# Paths – change if needed
# ---------------------------------------------------------------------------
IN_DIR = "./04_clean_tables"                    # folder containing the raw .parquet files
OUT_DIR = "./cleaned_Yasi"           # output folder (keeps original files untouched)

os.makedirs(OUT_DIR, exist_ok=True)


def safe_strip(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """Strip whitespace only on columns that actually exist."""
    for c in cols:
        if c in df.columns:
            df[c] = df[c].astype("string").str.strip()
    return df

In [7]:
# ===========================================================================
# 1. statistics.parquet  (most important table for aces, double faults, etc.)
# ===========================================================================
print("Cleaning statistics ...")

stats = pd.read_parquet(f"{IN_DIR}/statistics.parquet", engine="fastparquet")

# Remove exact duplicate stat lines
stats = stats.drop_duplicates(
    subset=["match_id", "period", "statistic_name", "value_type"]
).reset_index(drop=True)

# Clean text columns
stats = safe_strip(stats, [
    "period", "statistic_category_name", "statistic_name",
    "home_stat", "away_stat", "statistic_type", "value_type"
])


Cleaning statistics ...


In [8]:
# ---------------------------------------------------------------------------
# Percentage columns – ONLY for value_type == "team"
# (value_type == "event" has no denominator → home_total / away_total are null)
# ---------------------------------------------------------------------------
with np.errstate(divide="ignore", invalid="ignore"):
    is_team = stats["value_type"] == "team"

    stats["home_pct"] = np.where(
        is_team & (stats["home_total"] > 0),
        (100 * stats["home_value"] / stats["home_total"]).round(1),
        np.nan
    )
    stats["away_pct"] = np.where(
        is_team & (stats["away_total"] > 0),
        (100 * stats["away_value"] / stats["away_total"]).round(1),
        np.nan
    )

In [9]:
# Optional but useful: numeric versions of the display strings
# (home_stat / away_stat are strings like "12", "57/101 (56%)")
stats["home_value_num"] = pd.to_numeric(stats["home_value"], errors="coerce")
stats["away_value_num"] = pd.to_numeric(stats["away_value"], errors="coerce")

# Sanity checks
assert stats.duplicated(subset=["match_id", "period", "statistic_name", "value_type"]).sum() == 0
print(f"  → statistics: {stats.shape[0]:,} rows | "
      f"null home_total: {stats['home_total'].isna().sum():,} | "
      f"value_type counts: {stats['value_type'].value_counts().to_dict()}")

stats.to_parquet(f"{OUT_DIR}/statistics.parquet", index=False)

  → statistics: 662,810 rows | null home_total: 402,480 | value_type counts: {'event': 402480, 'team': 260330}


In [10]:
# ===========================================================================
# 2. pbp.parquet — point-by-point
# ===========================================================================
print("Cleaning pbp ...")

pbp = pd.read_parquet(f"{IN_DIR}/pbp.parquet", engine="fastparquet")

pbp = pbp.drop_duplicates(
    subset=["match_id", "set_id", "game_id", "point_id"]
).reset_index(drop=True)

pbp = safe_strip(pbp, ["home_point", "away_point"])

# Helpful flag: tie-break points usually show pure numbers (0,1,2,...) 
# while normal games use 0/15/30/40/A
pbp["is_tiebreak_point"] = (
    pbp["home_point"].str.fullmatch(r"\d+") & 
    pbp["away_point"].str.fullmatch(r"\d+")
)

print(f"  → pbp: {pbp.shape[0]:,} rows | "
      f"tiebreak points: {pbp['is_tiebreak_point'].sum():,}")

pbp.to_parquet(f"{OUT_DIR}/pbp.parquet", index=False)

Cleaning pbp ...
  → pbp: 1,251,104 rows | tiebreak points: 1,137,453


In [11]:
# ===========================================================================
# 3. power.parquet — momentum
# ===========================================================================
print("Cleaning power ...")

power = pd.read_parquet(f"{IN_DIR}/power.parquet", engine="fastparquet")

power = power.drop_duplicates(
    subset=["match_id", "set_num", "game_num"]
).reset_index(drop=True)

# Already very clean – just assert
assert power.isnull().sum().sum() == 0
assert power.duplicated(subset=["match_id", "set_num", "game_num"]).sum() == 0

print(f"  → power: {power.shape[0]:,} rows | no nulls")

power.to_parquet(f"{OUT_DIR}/power.parquet", index=False)

Cleaning power ...
  → power: 228,732 rows | no nulls


In [12]:
# ===========================================================================
# 4. odds.parquet  (second most important)
# ===========================================================================
print("Cleaning odds ...")

odds = pd.read_parquet(f"{IN_DIR}/odds.parquet", engine="fastparquet")

# Fix the typo in the source column name
odds = odds.rename(columns={"winnig": "winning"})

# ---------------------------------------------------------------------------
# Convert "winning" to proper nullable boolean
# Source contains: True / False / None (and sometimes string versions)
# ---------------------------------------------------------------------------
odds["winning"] = odds["winning"].map({
    True: True, False: False,
    "True": True, "False": False,
    "true": True, "false": False
})
odds["winning"] = odds["winning"].astype("boolean")   # supports <NA>

# Clean text columns
odds = safe_strip(odds, [
    "market_name", "initial_fractional_value",
    "fractional_value", "choice_name"
])

Cleaning odds ...


In [13]:
# ---------------------------------------------------------------------------
# Convert fractional odds → decimal odds
# Examples: "9/4" → 3.25, "1/3" → 1.3333, "0/0" → NaN
# ---------------------------------------------------------------------------
def frac_to_decimal(s) -> float:
    if pd.isna(s) or s in ("", "0/0", "None"):
        return np.nan
    try:
        parts = str(s).split("/")
        if len(parts) != 2:
            return np.nan
        num, den = float(parts[0]), float(parts[1])
        if den == 0:
            return np.nan
        return round(1 + num / den, 4)
    except (ValueError, TypeError, ZeroDivisionError):
        return np.nan

odds["decimal_odds"] = odds["fractional_value"].apply(frac_to_decimal)
odds["initial_decimal_odds"] = odds["initial_fractional_value"].apply(frac_to_decimal)

# Quick quality report
n_zero = (odds["fractional_value"] == "0/0").sum()
n_null_decimal = odds["decimal_odds"].isna().sum()
print(f"  → odds: {odds.shape[0]:,} rows | "
      f"'0/0' rows: {n_zero} | "
      f"null decimal_odds: {n_null_decimal} | "
      f"winning nulls: {odds['winning'].isna().sum()}")

odds.to_parquet(f"{OUT_DIR}/odds.parquet", index=False)

  → odds: 29,058 rows | '0/0' rows: 2 | null decimal_odds: 55 | winning nulls: 3102


In [14]:
# ===========================================================================
# 5. venue.parquet
# ===========================================================================
print("Cleaning venue ...")

venue = pd.read_parquet(f"{IN_DIR}/venue.parquet", engine="fastparquet")

venue = venue.drop_duplicates(subset=["match_id"]).reset_index(drop=True)

# The literal string "N/A" was used for city on the same rows where country is null
venue.loc[venue["city"] == "N/A", "city"] = pd.NA

venue = safe_strip(venue, ["city", "stadium", "country"])

# Optional normalized stadium name (helps grouping later)
venue["stadium_normalized"] = (
    venue["stadium"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

print(f"  → venue: {venue.shape[0]:,} rows | "
      f"country nulls: {venue['country'].isna().sum()} | "
      f"matches without venue row: {16873 - venue['match_id'].nunique()}")

venue.to_parquet(f"{OUT_DIR}/venue.parquet", index=False)

Cleaning venue ...
  → venue: 16,749 rows | country nulls: 83 | matches without venue row: 124


In [15]:
# ===========================================================================
# 6. votes.parquet
# ===========================================================================
print("Cleaning votes ...")

votes = pd.read_parquet(f"{IN_DIR}/votes.parquet", engine="fastparquet")

votes = votes.drop_duplicates(subset=["match_id"]).reset_index(drop=True)

# Already clean
assert votes.isnull().sum().sum() == 0
assert ((votes[["home_vote", "away_vote"]] < 0).any().any()) == False

print(f"  → votes: {votes.shape[0]:,} rows | no nulls, no negatives")

votes.to_parquet(f"{OUT_DIR}/votes.parquet", index=False)

Cleaning votes ...
  → votes: 16,873 rows | no nulls, no negatives


In [16]:
# ===========================================================================
# 7. season.parquet
# ===========================================================================
print("Cleaning season ...")

season = pd.read_parquet(f"{IN_DIR}/season.parquet", engine="fastparquet")

season = season.drop_duplicates(subset=["match_id"]).reset_index(drop=True)
season = safe_strip(season, ["name"])
season["year"] = season["year"].astype("Int64")

print(f"  → season: {season.shape[0]:,} rows | years: {sorted(season['year'].dropna().unique())}")

season.to_parquet(f"{OUT_DIR}/season.parquet", index=False)

Cleaning season ...
  → season: 16,873 rows | years: [np.int64(2024)]


In [17]:
print(f"Cleaned files are in: {os.path.abspath(OUT_DIR)}")

Cleaned files are in: D:\Tennis Project\Tennis Schema\tennis_project\cleaned_Yasi
